# MeanReversion backtest with Portfolio Optimization

In the chapter 04, we introduced `zipline` to simulate the computation of alpha factors from trailing cross-sectional market, fundamental, and alternative data. 

Now we will exploit the alpha factors to derive and act on buy and sell signals using the custom MeanReversion factor developed in the last chapter.

## Imports

In [ ]:
# Suppress warnings to keep output clean
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Standard library imports
import sys
import logging  # Using standard logging instead of deprecated logbook
from pytz import UTC

# Data manipulation and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Zipline imports for backtesting framework
# Note: If using zipline-reloaded, install with: pip install zipline-reloaded
from zipline import run_algorithm
from zipline.api import (
    attach_pipeline,        # Attach data pipeline to algorithm
    date_rules,             # Define when algo runs (e.g., weekly)
    time_rules,             # Define what time algo runs (e.g., market open)
    get_datetime,           # Get current datetime in backtest
    order_target_percent,   # Order to reach target portfolio %
    pipeline_output,        # Get pipeline results
    record,                 # Record custom data for analysis
    schedule_function,      # Schedule when functions run
    get_open_orders,        # Check for pending orders
    calendars,              # Trading calendar definitions
    set_commission,         # Set commission model
    set_slippage            # Set slippage model
)
from zipline.finance import commission, slippage
from zipline.pipeline import Pipeline, CustomFactor
from zipline.pipeline.factors import Returns, AverageDollarVolume

# Portfolio optimization library
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models, objective_functions
from pypfopt import expected_returns
from pypfopt.exceptions import OptimizationError

# Performance analysis library
# Note: pyfolio might need: pip install pyfolio-reloaded
try:
    from pyfolio.utils import extract_rets_pos_txn_from_zipline
except ImportError:
    # If pyfolio not available, we'll handle this later
    print("Warning: pyfolio not found. Install with: pip install pyfolio-reloaded")

# Setup basic logging instead of deprecated logbook
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s]: %(levelname)s: %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('Algorithm')

In [3]:
sns.set_style('whitegrid')

## Logging Setup

In [ ]:
# Logging is now configured in the imports cell above
# The deprecated logbook library has been replaced with standard Python logging

## Algo Settings

In [ ]:
# Algorithm Configuration Parameters
# These settings control the trading strategy behavior

MONTH = 21  # Approximate number of trading days in a month
YEAR = 12 * MONTH  # Number of trading days in a year (~252)

N_LONGS = 50  # Number of stocks to buy (long positions)
N_SHORTS = 50  # Number of stocks to short (short positions)

MIN_POS = 5  # Minimum number of positions required to run optimization
            # If we have fewer candidates, skip optimization

VOL_SCREEN = 1000  # Universe size: top N stocks by dollar volume
                   # Focuses on liquid stocks to avoid execution issues

In [ ]:
# Backtesting Period Configuration
# Define the simulation timeframe and starting capital

start = pd.Timestamp('2013-01-01', tz=UTC)  # Backtest start date (timezone-aware)
end = pd.Timestamp('2017-01-01', tz=UTC)    # Backtest end date (4 years)
capital_base = 1e7  # Starting capital: $10 million

## Mean Reversion Factor

In [ ]:
class MeanReversion(CustomFactor):
    """
    Custom Zipline factor that identifies mean reversion opportunities.
    
    Strategy Logic:
    - Calculates monthly returns over the past year
    - Compares latest monthly return to 12-month average
    - Normalizes by standard deviation (z-score)
    
    High positive values = recent underperformance → potential BUY (reversion up)
    High negative values = recent outperformance → potential SHORT (reversion down)
    """
    inputs = [Returns(window_length=MONTH)]  # Use monthly returns as input
    window_length = YEAR  # Look back 1 year of monthly returns
    
    def compute(self, today, assets, out, monthly_returns):
        """
        Calculate mean reversion z-score for each asset.
        
        Formula: (latest_return - mean_return) / std_return
        """
        df = pd.DataFrame(monthly_returns)
        # Get latest monthly return, subtract mean, divide by std dev
        factor = df.iloc[-1].sub(df.mean()).div(df.std())
        out[:] = factor

## Create Pipeline

The Pipeline created by the `compute_factors()` method returns a table with a long and a short column for the 25 stocks with the largest negative and positive deviations of their last monthly return from its annual average, normalized by the standard deviation. It also limited the universe to the 500 stocks with the highest average trading volume over the last 30 trading days. 

In [ ]:
def compute_factors():
    """
    Create the Zipline Pipeline that generates daily trading signals.
    
    Pipeline Process:
    1. Calculate mean reversion factor for all stocks
    2. Filter universe to top 1000 stocks by 30-day dollar volume
    3. Select top N_LONGS stocks with most negative z-scores (underperformers)
    4. Select top N_SHORTS stocks with most positive z-scores (overperformers)
    5. Return boolean flags for longs/shorts and numerical rankings
    
    Returns:
        Pipeline with columns:
        - 'longs': Boolean, True for stocks to buy
        - 'shorts': Boolean, True for stocks to short
        - 'ranking': Numerical rank of mean reversion factor
    """
    mean_reversion = MeanReversion()
    dollar_volume = AverageDollarVolume(window_length=30)
    
    return Pipeline(
        columns={
            'longs': mean_reversion.bottom(N_LONGS),   # Bottom N = most negative = underperformers
            'shorts': mean_reversion.top(N_SHORTS),    # Top N = most positive = overperformers
            'ranking': mean_reversion.rank(ascending=False)
        },
        screen=dollar_volume.top(VOL_SCREEN)  # Only consider liquid stocks
    )

`Before_trading_start()` ensures the daily execution of the pipeline and the recording of the results, including the current prices.

In [ ]:
def before_trading_start(context, data):
    """
    Runs automatically before market open each day.
    
    Tasks:
    1. Execute the factor pipeline to get today's signals
    2. Store results in context.factor_data for use during trading
    3. Record factor rankings for later analysis
    4. Record current prices for all candidate stocks
    
    Args:
        context: Stores state across algorithm execution
        data: Provides market data access
    """
    # Get today's pipeline output (longs, shorts, rankings)
    context.factor_data = pipeline_output('factor_pipeline')
    
    # Record factor rankings for performance analysis
    record(factor_data=context.factor_data.ranking)
    
    # Get all candidate assets
    assets = context.factor_data.index
    
    # Record current prices for all assets
    record(prices=data.current(assets, 'price'))

## Set up Rebalancing

The new `rebalance()` method submits trade orders to the `exec_trades()` method for the assets flagged for long and short positions by the pipeline with equal positive and negative weights. 

It also divests any current holdings that are no longer included in the factor signals:

In [ ]:
def exec_trades(data, positions):
    """
    Execute trades to reach target portfolio positions.
    
    For each asset and target weight:
    1. Check if asset is tradeable (market open, not halted, etc.)
    2. Check if there are no pending orders for this asset
    3. Submit order to reach target portfolio percentage
    
    Args:
        data: Market data accessor
        positions: Dict of {asset: target_percent} where target_percent is
                   the desired portfolio weight (0.0 to 1.0 for longs,
                   0.0 to -1.0 for shorts, 0.0 to exit)
    """
    for asset, target_percent in positions.items():
        # Only trade if market is open and no pending orders
        if data.can_trade(asset) and not get_open_orders(asset):
            # Order to reach target portfolio percentage
            # E.g., target_percent=0.05 means 5% of portfolio value
            order_target_percent(asset, target_percent)

In [ ]:
def rebalance(context, data):
    """
    Main rebalancing function - runs weekly at market open.
    
    Rebalancing Process:
    1. Get today's factor signals (longs and shorts)
    2. Exit any positions not in current signals
    3. Get 1 year of price history for optimization
    4. Optimize portfolio weights using mean-variance optimization
    5. Execute trades for optimized long and short portfolios
    6. Handle any errors by logging and keeping existing positions
    
    Args:
        context: Algorithm state storage
        data: Market data accessor
    """
    # Get today's pipeline results
    factor_data = context.factor_data
    assets = factor_data.index
    
    # Get stocks flagged for long and short positions
    longs = assets[factor_data.longs]
    shorts = assets[factor_data.shorts]
    
    # Find positions we currently hold but are no longer in signals
    # These need to be divested (sold/covered)
    divest = context.portfolio.positions.keys() - longs.union(shorts)
    exec_trades(data, positions={asset: 0 for asset in divest})
    
    # Log current portfolio value
    log.info('{} | {:11,.0f}'.format(
        get_datetime().date(), 
        context.portfolio.portfolio_value
    ))
    
    # Get historical prices for portfolio optimization
    # Need 252 trading days (1 year) of returns for covariance calculation
    prices = data.history(
        assets, 
        fields='price',
        bar_count=252+1,  # +1 because we need 252 returns from 253 prices
        frequency='1d'
    )
    
    # Only optimize if we have enough candidate stocks
    # (Optimization can fail with too few assets)
    if len(longs) > MIN_POS and len(shorts) > MIN_POS:
        try:
            # Optimize weights for long portfolio (maximize Sharpe ratio)
            long_weights = optimize_weights(prices.loc[:, longs])
            
            # Optimize weights for short portfolio (also maximize Sharpe, then invert)
            short_weights = optimize_weights(prices.loc[:, shorts], short=True)
            
            # Execute trades with optimized weights
            exec_trades(data, positions=long_weights)
            exec_trades(data, positions=short_weights)
            
        except Exception as e:
            # Log optimization errors but continue (keep existing positions)
            log.warning('{} {}'.format(get_datetime().date(), e))
    
    # Final safety check: exit any remaining positions not in current signals
    # This handles edge cases where positions slip through
    divest_pf = {asset: 0 for asset in context.portfolio.positions.keys()}
    exec_trades(data, positions=divest_pf)

## Optimize Portfolio Weights

In [ ]:
def optimize_weights(prices, short=False):
    """
    Calculate optimal portfolio weights using Mean-Variance Optimization.
    
    Optimization Method: Maximize Sharpe Ratio
    - Sharpe Ratio = (Expected Return - Risk Free Rate) / Volatility
    - Higher Sharpe = Better risk-adjusted returns
    
    Process:
    1. Calculate expected returns from historical mean
    2. Calculate covariance matrix from historical prices
    3. Use PyPortfolioOpt to find weights that maximize Sharpe ratio
    4. Clean weights (remove tiny allocations, normalize to sum to 1)
    5. If short portfolio, invert weights to negative
    
    Args:
        prices: DataFrame of historical prices (rows=dates, cols=assets)
        short: If True, return negative weights for short positions
        
    Returns:
        Dict of {asset: weight} where weights sum to 1.0 (or -1.0 for shorts)
    """
    # Calculate expected returns using historical mean
    # frequency=252 annualizes the returns (252 trading days/year)
    returns = expected_returns.mean_historical_return(
        prices=prices, 
        frequency=252
    )
    
    # Calculate covariance matrix (measures how assets move together)
    # Used to estimate portfolio volatility
    cov = risk_models.sample_cov(
        prices=prices, 
        frequency=252
    )
    
    # Create Efficient Frontier optimizer
    # weight_bounds=(0,1) means long-only for this sub-portfolio
    # Each sub-portfolio is optimized separately, then shorts are inverted
    ef = EfficientFrontier(
        expected_returns=returns,
        cov_matrix=cov,
        weight_bounds=(0, 1),  # 0% to 100% per asset
        solver='SCS'  # SCS solver is more robust than default ECOS
    )
    
    # Find portfolio weights that maximize Sharpe ratio
    ef.max_sharpe()
    
    # Clean and return weights
    if short:
        # For short portfolio, invert weights to negative
        return {asset: -weight for asset, weight in ef.clean_weights().items()}
    else:
        # For long portfolio, return positive weights
        return ef.clean_weights()

## Initialize Backtest

The `rebalance()` method runs according to `date_rules` and `time_rules` set by the `schedule_function()` utility at the beginning of the week, right after `market_open` as stipulated by the built-in `US_EQUITIES` calendar (see docs for details on rules). 

You can also specify a trade commission both in relative terms and as a minimum amount. There is also an option to define slippage, which is the cost of an adverse change in price between trade decision and execution

In [ ]:
def initialize(context):
    """
    Zipline algorithm initialization - runs once at start of backtest.
    
    Setup Tasks:
    1. Attach the factor pipeline to run daily
    2. Schedule rebalancing function to run weekly
    3. Set commission and slippage models for realistic costs
    
    Args:
        context: Stores algorithm state throughout backtest
    """
    # Attach our factor pipeline with name 'factor_pipeline'
    # This will compute signals daily and be available in before_trading_start
    attach_pipeline(compute_factors(), 'factor_pipeline')
    
    # Schedule rebalancing to run weekly at market open
    # date_rules.week_start() = Monday (or first trading day of week)
    # time_rules.market_open() = Right after market opens (9:31 AM ET)
    # calendar = US_EQUITIES uses NYSE/NASDAQ trading calendar
    schedule_function(
        rebalance,
        date_rules.week_start(),
        time_rules.market_open(),
        calendar=calendars.US_EQUITIES
    )
    
    # Set realistic trading costs
    # Commission: $0.00075 per share, minimum $0.01 per trade
    # This models typical broker fees (e.g., Interactive Brokers)
    set_commission(
        us_equities=commission.PerShare(cost=0.00075, min_trade_cost=.01)
    )
    
    # Set slippage model (cost of moving the market with our trades)
    # volume_limit=0.0025 = Can't trade more than 0.25% of daily volume
    # price_impact=0.01 = 1% price impact per 1% of volume traded
    set_slippage(
        us_equities=slippage.VolumeShareSlippage(
            volume_limit=0.0025, 
            price_impact=0.01
        )
    )

## Run Algorithm

The algorithm executes upon calling the `run_algorithm()` function and returns the backtest performance `DataFrame`.

In [ ]:
# Run the backtest simulation
# 
# IMPORTANT: Data Bundle Setup Required
# The 'quandl' bundle may no longer be available. You'll need to:
#
# Option 1: Use a different bundle (if you have one ingested)
#   - Run: zipline bundles
#   - Replace 'quandl' below with your bundle name
#
# Option 2: Ingest the quandl bundle (requires API key)
#   - Get free API key from: https://data.nasdaq.com/
#   - Set environment variable: export QUANDL_API_KEY=your_key_here
#   - Run: zipline ingest -b quandl
#
# Option 3: Use zipline-reloaded with built-in data
#   - Install: pip install zipline-reloaded
#   - Some bundles may work without manual ingestion
#
# If you get "No bundle registered with name 'quandl'" error,
# follow one of the options above.

backtest = run_algorithm(
    start=start,                           # Start date of backtest
    end=end,                               # End date of backtest
    initialize=initialize,                 # Initialization function
    before_trading_start=before_trading_start,  # Pre-market function
    bundle='quandl',                       # Data bundle (may need to change)
    capital_base=capital_base              # Starting capital
)

## Extract pyfolio Inputs

The `extract_rets_pos_txn_from_zipline` utility provided by `pyfolio` extracts the data used to compute performance metrics.

In [ ]:
# Extract performance data from backtest results
# 
# This utility function processes the Zipline backtest output and extracts:
# - returns: Daily portfolio returns (pandas Series)
# - positions: Daily positions held (pandas DataFrame)
# - transactions: All trades executed (pandas DataFrame)
#
# These are used for detailed performance analysis with pyfolio

returns, positions, transactions = extract_rets_pos_txn_from_zipline(backtest)

## Persist Results for use with `pyfolio`

In [ ]:
# Save backtest results to HDF5 file for later analysis
# 
# HDF5 format is efficient for storing large DataFrames
# Structure:
#   /returns/pf_opt - Daily returns from this portfolio optimization strategy
#   /transactions/pf_opt - All trades from this strategy
#
# These can be compared with other strategies (like equal_weight)

with pd.HDFStore('backtests.h5') as store:
    store.put('returns/pf_opt', returns)
    store.put('transactions/pf_opt', transactions)

In [ ]:
# Load results from multiple strategies for comparison
#
# NOTE: This cell assumes you have run an equal_weight strategy previously
# and saved it to the same backtests.h5 file. If you get a KeyError,
# it means the equal_weight results don't exist yet.
#
# To run this comparison, you would need to:
# 1. Modify the rebalance() function to use equal weights instead of optimization
# 2. Run the backtest again
# 3. Save to 'returns/equal_weight' and 'transactions/equal_weight'
# 4. Then run this cell to compare both strategies

with pd.HDFStore('backtests.h5') as store:
    returns_pf = store['returns/pf_opt']          # Portfolio optimization results
    tx_pf = store['transactions/pf_opt']          
    returns_ew = store['returns/equal_weight']    # Equal weight results (may not exist)
    tx_ew = store['transactions/equal_weight']

## Plot Results

In [ ]:
# Plot backtest results: returns and transaction volume
#
# Top plot: Cumulative returns over time
#   - Shows how $1 invested would grow
#   - subtract(1) to start at 0% instead of 100%
#
# Bottom plot: Cumulative transaction volume
#   - Shows total dollar volume traded over time
#   - Grouped by day to get daily totals

fig, axes = plt.subplots(nrows=2, figsize=(14, 6))

# Plot cumulative returns
returns.add(1).cumprod().sub(1).plot(
    ax=axes[0], 
    title='Cumulative Returns'
)

# Plot cumulative transactions
# Fixed: .dt.day instead of .dt.dt.day (pandas datetime accessor)
transactions.groupby(transactions.dt.day).txn_dollars.sum().cumsum().plot(
    ax=axes[1], 
    title='Cumulative Transactions'
)

sns.despine()
fig.tight_layout()

In [ ]:
# Compare two strategies side-by-side: Equal Weight vs Portfolio Optimization
#
# Layout: 2x2 grid
#   Left column: Cumulative returns
#   Right column: Cumulative transaction volume
#   Top row: Equal weight strategy
#   Bottom row: Mean-variance optimization strategy
#
# This visualization helps compare:
# - Which strategy performs better (higher returns)
# - Which strategy trades more (higher transaction costs)

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(16, 8), sharey='col')

# Equal Weight - Returns
returns_ew.add(1).cumprod().sub(1).plot(
    ax=axes[0][0],
    title='Cumulative Returns - Equal Weight'
)

# Portfolio Optimization - Returns
returns_pf.add(1).cumprod().sub(1).plot(
    ax=axes[1][0],
    title='Cumulative Returns - Mean-Variance Optimization'
)

# Equal Weight - Transactions
# Fixed: .dt.day instead of .dt.dt.day (pandas datetime accessor)
tx_ew.groupby(tx_ew.dt.day).txn_dollars.sum().cumsum().plot(
    ax=axes[0][1],
    title='Cumulative Transactions - Equal Weight'
)

# Portfolio Optimization - Transactions
# Fixed: .dt.day instead of .dt.dt.day (pandas datetime accessor)
tx_pf.groupby(tx_pf.dt.day).txn_dollars.sum().cumsum().plot(
    ax=axes[1][1],
    title='Cumulative Transactions - Mean-Variance Optimization'
)

fig.suptitle('Equal Weight vs Mean-Variance Optimization', fontsize=16)
sns.despine()
fig.tight_layout()
fig.subplots_adjust(top=.9)